# Laboratório da Silver

A exploração da Bronze (`exploracao_bronze.ipynb`) terminou com um checklist de 7 tratamentos. Antes de escrever o código de produção da Silver, prototipo cada um aqui — a regra é: **nenhuma transformação vira script sem ter sido validada neste notebook**, com contagem antes/depois.

O que vou prototipar:

1. decodificação de códigos usando a tabela `dicionario` da fonte;
2. casting (tudo chega como texto na Bronze);
3. flags de ausência (o nulo de proficiência é informação, não defeito);
4. derivação da UF a partir do código IBGE do município;
5. integração com as metas (e o teste de "o join não explode linhas");
6. empilhamento das 3 tabelas de metas numa estrutura única;
7. quarentena do município órfão + a suite de data quality de produção.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import data_quality as dq

BRONZE = Path("../data/bronze/batch")
pd.set_option("display.max_columns", 40)

alunos = pd.read_parquet(BRONZE / "alunos")
municipio = pd.read_parquet(BRONZE / "municipio")
uf = pd.read_parquet(BRONZE / "uf")
metas_brasil = pd.read_parquet(BRONZE / "meta_alfabetizacao_brasil")
metas_uf = pd.read_parquet(BRONZE / "meta_alfabetizacao_uf")
metas_municipio = pd.read_parquet(BRONZE / "meta_alfabetizacao_municipio")
dicionario = pd.read_parquet(BRONZE / "dicionario")

print(f"alunos: {alunos.shape[0]:,} linhas")

alunos: 3,867,999 linhas


## 1. Decodificação via `dicionario`

A fonte publica um de-para oficial: `(id_tabela, nome_coluna, chave) → valor`. É ele que diz o que significam os códigos de `rede`, `presenca` etc. — melhor do que eu chutar.

In [2]:
dicionario[dicionario["id_tabela"] == "alunos"][["nome_coluna", "chave", "valor"]]

,nome_coluna,chave,valor
0,alfabetizado,0,Não
1,alfabetizado,1,Sim
2,preenchimento_caderno,0,Prova não preenchida
3,preenchimento_caderno,1,Prova preenchida
4,presenca,0,Ausente
5,presenca,1,Presente
10,rede,1,Federal
11,rede,2,Estadual
16,rede,3,Municipal
18,rede,4,Privada


In [3]:
def mapa(tabela, coluna):
    d = dicionario.query("id_tabela == @tabela and nome_coluna == @coluna")
    return dict(zip(d["chave"], d["valor"]))

alunos["rede_nome"] = alunos["rede"].map(mapa("alunos", "rede"))
alunos["presenca_nome"] = alunos["presenca"].map(mapa("alunos", "presenca"))

print("códigos sem rótulo:", alunos[["rede_nome", "presenca_nome"]].isna().sum().sum())
alunos["rede_nome"].value_counts()

códigos sem rótulo: 0


rede_nome
Municipal    3432576
Estadual      435398
Privada           25
Name: count, dtype: int64

Cobertura total: todo código presente nos microdados tem rótulo no dicionário. A rede federal (código 1) simplesmente não aparece nos microdados — e os 25 alunos de rede privada são resíduo estatístico, dado que a avaliação foca a rede pública. A decodificação entra na Silver exatamente assim.

## 2. Casting

Tudo veio como texto da Bronze. Antes de converter, o teste que importa: existe algum valor **não numérico** escondido nessas colunas? (Se `pd.to_numeric` gerar mais nulos do que a coluna já tinha, tem sujeira.)

In [4]:
cols_inteiras = ["id_municipio", "id_escola", "serie", "rede", "presenca",
                 "preenchimento_caderno", "alfabetizado"]

sujeira = {
    c: int(pd.to_numeric(alunos[c], errors="coerce").isna().sum() - alunos[c].isna().sum())
    for c in cols_inteiras
}
sujeira

{'id_municipio': 0,
 'id_escola': 0,
 'serie': 0,
 'rede': 0,
 'presenca': 0,
 'preenchimento_caderno': 0,
 'alfabetizado': 0}

Nenhum valor não numérico escondido em coluna nenhuma — a conversão é segura, sem regra de limpeza extra. Vou de `Int64` (inteiro anulável do pandas) para conviver com nulos legítimos, e o `ano` volta de categoria (efeito colateral da partição Hive) para inteiro:

In [5]:
for c in cols_inteiras:
    alunos[c] = pd.to_numeric(alunos[c]).astype("Int64")
alunos["ano"] = alunos["ano"].astype(int)   # veio como categoria da partição Hive

alunos.dtypes

id_municipio               Int64
id_escola                  Int64
id_aluno                     str
caderno                      str
serie                      Int64
rede                       Int64
presenca                   Int64
preenchimento_caderno      Int64
alfabetizado               Int64
proficiencia             float64
peso_aluno               float64
_row_hash                 uint64
_ingestion_ts                str
_source                      str
ano                        int64
rede_nome                    str
presenca_nome                str
dtype: object

## 3. Flags de ausência

Achado da EDA: proficiência nula = aluno ausente (mais um resíduo de presentes sem nota). Em vez de imputar ou descartar, materializo isso em duas flags — quem consome a Silver decide o recorte com clareza.

In [6]:
alunos["presente"] = alunos["presenca"] == 1
alunos["sem_nota"] = alunos["proficiencia"].isna()

pd.crosstab(alunos["presente"], alunos["sem_nota"],
            rownames=["presente"], colnames=["sem nota"])

sem nota,False,True
presente,,
False,0,512153
True,3354661,1185


## 4. UF a partir do código IBGE

Os microdados não trazem a UF, mas o código IBGE de município embute: os 2 primeiros dígitos são o código da unidade federativa. O mapa código→sigla é tabela pública do IBGE (estável desde os anos 70), então mantenho como constante no código — e valido a cobertura aqui.

In [7]:
UF_POR_CODIGO = {
    11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO",
    21: "MA", 22: "PI", 23: "CE", 24: "RN", 25: "PB", 26: "PE", 27: "AL", 28: "SE", 29: "BA",
    31: "MG", 32: "ES", 33: "RJ", 35: "SP",
    41: "PR", 42: "SC", 43: "RS",
    50: "MS", 51: "MT", 52: "GO", 53: "DF",
}

alunos["sigla_uf"] = (alunos["id_municipio"] // 100000).map(UF_POR_CODIGO)

siglas_fonte = set(uf["sigla_uf"].unique()) - {"Brasil"}
print("alunos sem UF derivada:", int(alunos["sigla_uf"].isna().sum()))
print("siglas derivadas fora da tabela uf:", set(alunos["sigla_uf"].unique()) - siglas_fonte)
alunos["sigla_uf"].value_counts().head()

alunos sem UF derivada: 0
siglas derivadas fora da tabela uf: {'DF'}


sigla_uf
SP    443777
MG    413557
RJ    311198
PR    277446
BA    253300
Name: count, dtype: int64

Cobertura de 100%: nenhum aluno ficou sem UF. E um achado de brinde: o `DF` existe nos microdados (Brasília), mas **não aparece na tabela `uf` da fonte** — por isso ela tem 26 unidades, não 27. Não é impedimento (a agregação por UF na Gold sai dos próprios microdados), mas vai registrado no dicionário de dados.

## 5. Integração com as metas

O join que a Gold vai precisar: microdados agregados por `(ano, id_municipio)` contra `meta_alfabetizacao_municipio`. Dois testes obrigatórios: o join **não pode multiplicar linhas** (a meta é única por município/ano — validei na sondagem) e quero saber a taxa de match.

In [8]:
metas_municipio["id_municipio"] = pd.to_numeric(metas_municipio["id_municipio"]).astype("Int64")
metas_municipio["ano"] = pd.to_numeric(metas_municipio["ano"]).astype(int)

chaves = alunos[["ano", "id_municipio"]].drop_duplicates()
teste = chaves.merge(
    metas_municipio[["ano", "id_municipio", "taxa_alfabetizacao"]],
    on=["ano", "id_municipio"], how="left", indicator=True,
)

print(f"combinações (ano, município) nos microdados: {len(chaves):,}")
print(f"linhas após o join: {len(teste):,}  (não pode ter crescido)")
teste["_merge"].value_counts()

combinações (ano, município) nos microdados: 10,392
linhas após o join: 10,392  (não pode ter crescido)


_merge
both          10048
left_only       344
right_only        0
Name: count, dtype: int64

Dois resultados bons: o join preservou as 10.392 linhas (chave única confirmada na prática) e 10.048 combinações têm meta. Sobram **344 sem par** (`left_only`). Hipótese: a meta municipal é definida para a rede municipal — municípios avaliados só pela rede estadual ficariam de fora; confirmo esse padrão na Silver. Para o contrato, o que importa é a decisão: o join com metas será `left` (nenhum aluno é descartado por falta de meta) e o caso "sem meta" fica explícito para a Gold tratar.

## 6. Metas numa estrutura única

As três tabelas de metas (Brasil, UF, município) têm o mesmo desenho de colunas — dá para empilhar com uma coluna `nivel` indicando o grão. O ponto de atenção que a sondagem revelou: aqui a `rede` vem como **texto** (`Pública`, `Municipal`), enquanto nas tabelas de resultado ela é **código** (`0`, `2`, `3`, `5`...). A Silver precisa padronizar para um vocabulário só.

In [9]:
metas = pd.concat(
    [
        metas_brasil.assign(nivel="brasil"),
        metas_uf.assign(nivel="uf"),
        metas_municipio.assign(nivel="municipio"),
    ],
    ignore_index=True,
)

print(metas.groupby("nivel").size())
print()
print("valores de rede nas metas:", metas["rede"].unique())
metas[metas["nivel"] == "uf"].head(3).drop(columns=["_row_hash", "_ingestion_ts", "_source"])

nivel
brasil           3
municipio    10704
uf              81
dtype: int64

valores de rede nas metas: <ArrowStringArray>
['Pública', 'Municipal']
Length: 2, dtype: str


,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,nivel,sigla_uf,id_municipio,nivel_alfabetizacao
3,2024,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,RR,<NA>,<NA>
4,2023,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,RR,<NA>,<NA>
5,2024,Pública,38.39,38.3,45.9,53.6,61.2,68.3,74.6,80.0,92.84,uf,SE,<NA>,<NA>


## 7. Quarentena e a suite de qualidade

A EDA achou um `id_municipio` órfão (`5219308`, prefixo 52 = GO). Estratégia da Silver: essas linhas não somem — vão para `silver/quarentena/` com o motivo registrado, e o pipeline segue com o resto. Quanto isso representa?

In [10]:
municipio["id_municipio"] = pd.to_numeric(municipio["id_municipio"]).astype("Int64")

na_quarentena = alunos[~alunos["id_municipio"].isin(municipio["id_municipio"])]
print(f"linhas afetadas: {len(na_quarentena):,} ({len(na_quarentena) / len(alunos):.4%} do total)")
na_quarentena[["ano", "id_municipio", "id_escola", "rede_nome", "presente"]].head()

linhas afetadas: 410 (0.0106% do total)


,ano,id_municipio,id_escola,rede_nome,presente
15255,2023,5219308,60042766,Municipal,False
20882,2023,5219308,60042766,Municipal,False
26220,2023,5219308,60042127,Municipal,False
27520,2023,5219308,60042126,Municipal,False
29625,2023,5219308,60042126,Municipal,False


São **410 linhas (0,01% da base)**, todas do código `5219308`. Custo mínimo para garantir a integridade referencial: em vez de sumirem num filtro, essas linhas vão para `silver/quarentena/` com a coluna `motivo` preenchida — continuam auditáveis, e dá para reprocessá-las se a dimensão de municípios for corrigida na fonte.

Para fechar, rodo os checks de qualidade **do módulo de produção** (`src/utils/data_quality.py`) — os mesmos que o script da Silver vai usar. A graça é ver o check de integridade referencial reprovar de propósito: é ele que justifica a quarentena. No script, a quarentena roda antes do check, então a Silver final passa limpa.

In [11]:
checks = [
    dq.check_not_null(alunos, "alunos", ["ano", "id_municipio", "id_aluno"]),
    dq.check_unique(alunos, "alunos", ["id_aluno", "ano"]),
    dq.check_range(alunos, "alunos", "proficiencia", 0, 1000),
    dq.check_referential_integrity(alunos, "alunos", "id_municipio", municipio, "id_municipio"),
]
pd.DataFrame(checks)

,entity,dimension,check,passed,status,severity,detail
0,alunos,completude,not_null,True,pass,critical,"sem nulos em ['ano', 'id_municipio', 'id_aluno']"
1,alunos,unicidade,"unique(id_aluno,ano)",True,pass,critical,0 duplicatas
2,alunos,validade,"range(proficiencia em [0, 1000])",True,pass,critical,0 valores fora do intervalo
3,alunos,consistencia,referential(id_municipio -> id_municipio),False,fail,critical,410 chaves órfãs


## 8. Normalização de texto

Revisando as anotações das aulas contra o que já prototipei, ficaram três pontas soltas — começo pela normalização de texto (o clássico "João" ≠ "joão"). No nosso dado ela se aplica aos rótulos de `rede`, que aparecem com três vocabulários diferentes: código nas tabelas de resultado, texto com acento e parênteses no dicionário e texto capitalizado nas metas. Um normalizador único resolve os três:

In [12]:
import re
import unicodedata


def normaliza(texto):
    s = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")


metas["rede_padronizada"] = metas["rede"].map(normaliza)
municipio["rede_nome"] = municipio["rede"].map(mapa("municipio", "rede")).map(normaliza)

print("metas:     ", sorted(metas["rede_padronizada"].unique()))
print("resultados:", sorted(municipio["rede_nome"].unique()))

metas:      ['municipal', 'publica']
resultados: ['estadual', 'municipal', 'publica_estadual_e_municipal', 'total_federal_estadual_municipal_e_privada']


Um normalizador só e os três vocabulários convergem: `municipal` já coincide dos dois lados. Restam dois ajustes semânticos para a Silver: o código 5 (`publica_estadual_e_municipal`) é o mesmo conceito da `publica` das metas — vão para o mesmo rótulo — e o código 0 vira `total`. Com isso, `rede` fala uma língua só do resultado à meta.

## 9. O peso amostral que faltava olhar

`peso_aluno` é a coluna que suspeito explicar a diferença entre o meu recálculo e a taxa oficial — e eu ainda não tinha olhado a cara dela. Checagem de sanidade: range, zeros, e se o nulo dela acompanha a ausência (como a proficiência):

In [13]:
peso = alunos["peso_aluno"]
print(peso.describe().round(3).to_string())
print("zerados ou negativos:", int((peso <= 0).sum()))

pd.crosstab(alunos["presente"], peso.isna(), rownames=["presente"], colnames=["peso nulo"])

count    3354661.000
mean           1.148
std            0.359
min            0.095
25%            1.001
50%            1.092
75%            1.208
max          142.545
zerados ou negativos: 0


peso nulo,False,True
presente,,
False,0,512153
True,3354661,1185


O peso se comporta bem: sempre positivo, mediana em ~1,09, nenhum zero. A cauda direita é longa (máximo de 142,5 contra 1,21 no terceiro quartil) — esperado para um peso amostral que corrige sub-representação, então não é outlier para "tratar", é característica para documentar. E o nulo do peso segue exatamente o padrão da proficiência: os 512.153 ausentes mais os mesmos 1.185 presentes sem nota. Para a Silver ficam dois encaminhamentos: check de range (`peso_aluno > 0`) e a confirmação, na Gold, de que a taxa oficial usa a média ponderada por esse peso.

## 10. Os checks que faltavam no framework

Última ponta solta: o módulo de qualidade não tinha validação de **formato** (regex), **consistência linha a linha** entre campos, nem **completude com threshold** — e o relatório era binário, sem o nível *warning* para problemas que merecem registro mas não justificam derrubar o pipeline. Adicionei os três ao `src/utils/data_quality.py` e testo aqui contra os microdados:

In [14]:
com_nota = alunos[alunos["presente"] & ~alunos["sem_nota"]]

checks_novos = [
    dq.check_format(alunos, "alunos", "id_municipio", r"\d{7}"),
    dq.check_format(alunos, "alunos", "id_escola", r"\d{8}"),
    dq.check_consistency(alunos, "alunos", "ausente implica sem nota",
                         ~(~alunos["presente"] & alunos["proficiencia"].notna())),
    dq.check_consistency(com_nota, "alunos", "alfabetizado bate com a regra dos 743",
                         com_nota["alfabetizado"] == (com_nota["proficiencia"] >= 743).astype(int)),
    dq.check_consistency(alunos, "alunos", "presente implica ter nota",
                         ~(alunos["presente"] & alunos["sem_nota"]), severity="warning"),
    dq.check_completeness(alunos, "alunos", "proficiencia", min_pct=80),
]
pd.DataFrame(checks_novos)[["dimension", "check", "status", "detail"]]

,dimension,check,status,detail
0,validade,format(id_municipio ~ /\d{7}/),pass,0 valores fora do padrão
1,validade,format(id_escola ~ /\d{8}/),pass,0 valores fora do padrão
2,consistencia,consistency(ausente implica sem nota),pass,0 violações
3,consistencia,consistency(alfabetizado bate com a regra dos ...,pass,0 violações
4,consistencia,consistency(presente implica ter nota),warning,1185 violações
5,completude,completeness(proficiencia >= 80%),pass,86.7% preenchido


Saiu exatamente como deveria: os formatos dos ids passam, a consistência do `alfabetizado` com a regra dos 743 fecha com **zero violações** — agora validada registro a registro, não só no agregado — e os 1.185 presentes sem nota aparecem como **warning**, não como falha: ficam registrados no relatório sem derrubar o pipeline. Era o comportamento que faltava no framework.

## Contrato da Silver (o que vai para o script)

Validei tudo o que precisava — incluindo as pontas soltas que a revisão das anotações de aula apontou. O `src/02_silver/tratamento_integracao.py` vai materializar:

| Saída | Conteúdo |
|---|---|
| `silver/alunos/` | microdados com casting, `rede_nome`/`presenca_nome` normalizados, flags `presente`/`sem_nota`, `sigla_uf` derivada; particionado por ano |
| `silver/quarentena/alunos/` | linhas reprovadas (município órfão), com coluna `motivo` |
| `silver/resultados/` | tabelas `municipio` e `uf` com casting e `rede` no vocabulário padronizado (`municipal`, `estadual`, `publica`, `total`...) |
| `silver/metas/` | as 3 tabelas empilhadas com `nivel`, `rede` no mesmo vocabulário |

Ordem que o script aplica: casting → decodificação + normalização de texto → flags → UF → quarentena → DQ formal → escrita particionada.

Suite de DQ da Silver (já com os checks novos do módulo):

- `not_null` nas chaves e `unique (id_aluno, ano)`;
- `range`: proficiência em [0, 1000] e `peso_aluno > 0`;
- `format`: `id_municipio` com 7 dígitos, `id_escola` com 8;
- `consistency`: ausente ⇒ sem nota; alfabetizado ⇔ proficiência ≥ 743;
- `referential`: `id_municipio` → dimensão municípios (roda **depois** da quarentena, então passa);
- warnings (não derrubam): presente ⇒ ter nota; completude de proficiência ≥ 80%.

Os números deste notebook são os valores de referência para validar a primeira execução do script.